## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from PIL import Image
from torchvision import transforms

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

ROOT = Path('..')
FIGURES_DIR = ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print('Figures dir:', FIGURES_DIR.resolve())

## 2. Dataset Loading

In [ ]:
from src.dataset import MVTecDataset

train_ds = MVTecDataset(ROOT, split='train')
val_ds   = MVTecDataset(ROOT, split='val')
test_ds  = MVTecDataset(ROOT, split='test')

print(train_ds)
print(val_ds)
print(test_ds)
print()

rows = []
for ds in [train_ds, val_ds, test_ds]:
    normal  = sum(1 for _, lbl in ds.samples if lbl == 0)
    anomaly = sum(1 for _, lbl in ds.samples if lbl == 1)
    rows.append({
        'split':   ds.split,
        'total':   len(ds),
        'normal':  normal,
        'anomaly': anomaly,
    })

df_counts = pd.DataFrame(rows, columns=['split', 'total', 'normal', 'anomaly'])
print(df_counts.to_string(index=False))

## 3. Normal Samples Grid

In [ ]:
_mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def denorm(tensor):
    """Reverse ImageNet normalisation -> HWC numpy in [0, 1]."""
    return (tensor * _std + _mean).clamp(0, 1).permute(1, 2, 0).numpy()

random.seed(42)
indices = random.sample(range(len(train_ds)), 16)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for ax, idx in zip(axes.flat, indices):
    img_tensor, _, fname = train_ds[idx]
    ax.imshow(denorm(img_tensor))
    ax.set_title(Path(fname).stem, fontsize=8)
    ax.axis('off')

plt.suptitle('Normal Training Samples (seed=42)', fontsize=14)
plt.tight_layout()
out_path = FIGURES_DIR / 'normal_samples.png'
fig.savefig(out_path, bbox_inches='tight')
print('Saved:', out_path.resolve())
plt.show()

## 4. Defect Samples Grid

In [ ]:
HAZELNUT_TEST = ROOT / 'data' / 'mvtec' / 'hazelnut' / 'test'
DEFECT_TYPES  = ['crack', 'cut', 'hole', 'print']

raw_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
])

fig, axes = plt.subplots(4, 4, figsize=(12, 12))

for row_idx, defect in enumerate(DEFECT_TYPES):
    defect_dir = HAZELNUT_TEST / defect
    img_files  = sorted(defect_dir.glob('*.png'))[:4]

    for col_idx, img_path in enumerate(img_files):
        ax  = axes[row_idx][col_idx]
        img = raw_transform(Image.open(img_path).convert('RGB'))
        ax.imshow(img)
        ax.set_title(img_path.stem, fontsize=7)
        ax.axis('off')

    # Row label on the first column
    axes[row_idx][0].set_ylabel(
        defect, fontsize=11, rotation=0, labelpad=55, va='center'
    )

plt.suptitle('Defect Samples by Type', fontsize=14)
plt.tight_layout()
out_path = FIGURES_DIR / 'defect_samples.png'
fig.savefig(out_path, bbox_inches='tight')
print('Saved:', out_path.resolve())
plt.show()

## 5. Pixel Intensity Histogram

In [ ]:
def sample_grayscale_pixels(img_paths, n=50, seed=42):
    """Open n images, convert to grayscale [0,1], return flattened pixel array."""
    rng = random.Random(seed)
    selected = rng.sample(list(img_paths), min(n, len(img_paths)))
    pixel_arrays = []
    for p in selected:
        arr = np.array(Image.open(p).convert('L'), dtype=np.float32) / 255.0
        pixel_arrays.append(arr.flatten())
    return np.concatenate(pixel_arrays)

normal_paths  = sorted((HAZELNUT_TEST / 'good').glob('*.png'))
anomaly_paths = []
for defect in DEFECT_TYPES:
    anomaly_paths.extend(sorted((HAZELNUT_TEST / defect).glob('*.png')))

normal_px  = sample_grayscale_pixels(normal_paths,  n=50)
anomaly_px = sample_grayscale_pixels(anomaly_paths, n=50)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(normal_px,  bins=100, alpha=0.6, color='blue', label='Normal',  density=True)
ax.hist(anomaly_px, bins=100, alpha=0.6, color='red',  label='Anomaly', density=True)
ax.axvline(
    normal_px.mean(), color='blue', linestyle='--', linewidth=1.5,
    label=f'Normal mean = {normal_px.mean():.3f}'
)
ax.axvline(
    anomaly_px.mean(), color='red', linestyle='--', linewidth=1.5,
    label=f'Anomaly mean = {anomaly_px.mean():.3f}'
)
ax.set_xlabel('Pixel Intensity (grayscale, normalised 0-1)')
ax.set_ylabel('Density')
ax.set_title('Pixel Intensity Distribution: Normal vs Anomalous')
ax.legend()
plt.tight_layout()
out_path = FIGURES_DIR / 'pixel_intensity_dist.png'
fig.savefig(out_path, bbox_inches='tight')
print('Saved:', out_path.resolve())
plt.show()

## 6. Image Size Verification

In [ ]:
HAZELNUT_TRAIN = ROOT / 'data' / 'mvtec' / 'hazelnut' / 'train' / 'good'
all_train_paths = sorted(HAZELNUT_TRAIN.glob('*.png'))

_crop = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
])

size_set = set()
for p in all_train_paths:
    img = _crop(Image.open(p).convert('RGB'))
    size_set.add(img.size)  # PIL size: (width, height)

n = len(all_train_paths)
if size_set == {(224, 224)}:
    print(f'All {n} images verified: 224x224')
else:
    print(f'WARNING — unexpected sizes found: {size_set}')